In [1]:
from termcolor import colored
import numpy as np
import random
import copy
import math
import sys

In [2]:
ROW_COUNT = 6
COLUMN_COUNT = 10

PLAYER1 = 0
PLAYER2 = 1

EMPTY = 0
PLAYER1_PIECE = 1
PLAYER2_PIECE = 2

WINDOW_LENGTH = 5

In [3]:
def create_board():
    board = np.zeros((ROW_COUNT, COLUMN_COUNT))
    return board


def print_board(board):
    flip_board = np.flip(board, 0)
    print()
    
    for r in range(ROW_COUNT):
        for c in range(COLUMN_COUNT):
            
            if flip_board[r][c] == PLAYER1_PIECE:
                print(colored(int(flip_board[r][c]), 'grey', 'on_red'), end=' ')
                
            elif flip_board[r][c] == PLAYER2_PIECE:
                print(colored(int(flip_board[r][c]), 'grey', 'on_blue'), end=' ')
                
            else:
                print(colored(int(flip_board[r][c]), 'grey'), end=' ')
                
        print()


# Logic
def get_next_open_row(board, col):
    for r in range(ROW_COUNT):
        if board[r][col] == 0:
            return r


def drop_piece(board, row, col, piece):
    board[row][col] = piece


def is_valid_location(board, col):
    if col != None:
        return board[ROW_COUNT - 1][col] == 0
    return False


def get_valid_locations(board):
    valid_locations = []
    for col in range(COLUMN_COUNT):
        if is_valid_location(board, col):
            valid_locations.append(col)
    return valid_locations


def is_terminal_node(board):
    return winning_move(board, PLAYER1_PIECE) or winning_move(board, PLAYER2_PIECE) or \
           len(get_valid_locations(board)) == 0


def winning_move(board, piece):
    # Check horizontal locations for win
    for c in range(COLUMN_COUNT - 4):
        for r in range(ROW_COUNT):
            if board[r][c] == piece and board[r][c + 1] == piece and board[r][c + 2] == piece and \
                    board[r][c + 3] == piece and board[r][c + 4] == piece:
                return True

    # Check vertical locations for win
    for c in range(COLUMN_COUNT):
        for r in range(ROW_COUNT - 4):
            if board[r][c] == piece and board[r + 1][c] == piece and board[r + 2][c] == piece and \
                    board[r + 3][c] == piece and board[r + 4][c] == piece:
                return True

    # Check positively sloped diaganols
    for c in range(COLUMN_COUNT - 4):
        for r in range(ROW_COUNT - 4):
            if board[r][c] == piece and board[r + 1][c + 1] == piece and board[r + 2][c + 2] == piece and \
                    board[r + 3][c + 3] == piece and board[r + 4][c + 4] == piece:
                return True
            
    # Check negatively sloped diaganols
    for c in range(COLUMN_COUNT - 4):
        for r in range(4, ROW_COUNT):
            if board[r][c] == piece and board[r - 1][c + 1] == piece and board[r - 2][c + 2] == piece and \
                    board[r - 3][c + 3] == piece and board[r - 4][c + 4] == piece:
                return True
            
    return False

In [23]:
def evaluate_window(window, piece):
# This function attempts to evaluate a specific part of the board called window for a specific player. 
# So it counts the pieces of the player and also its oponent in that window and returns a score based on that. 
    score = 0
    competitor_player_piece = PLAYER2_PIECE# YOU SHOULD DO THIS LINE. The competitor piece is 1 or 2?
    if piece == PLAYER2_PIECE:
        competitor_player_piece =PLAYER1_PIECE # YOU SHOULD DO THIS LINE.
#         first
#     if piece == PLAYER1_PIECE:
#         if window.count(piece) == 5:
#             score=20 # YOU SHOULD DO THIS LINE.
#         elif window.count(piece) == 4 and window.count(EMPTY) == 1:
#             score=18 # YOU SHOULD DO THIS LINE.
#         elif window.count(piece) == 3 and window.count(EMPTY) == 2:
#             score=10# YOU SHOULD DO THIS LINE.
#         elif window.count(competitor_player_piece) == 4 and window.count(EMPTY) == 1:
#             score=16# YOU SHOULD DO THIS LINE.
#         elif window.count(EMPTY)==0:
#             score=0
#         else:
#             score =window.count(piece)-window.count(competitor_player_piece)
#     elif piece == PLAYER2_PIECE:
#         if window.count(piece) == 5:
#             score=-20 # YOU SHOULD DO THIS LINE.
#         elif window.count(piece) == 4 and window.count(EMPTY) == 1:
#             score=-18 # YOU SHOULD DO THIS LINE.
#         elif window.count(piece) == 3 and window.count(EMPTY) == 2:
#             score=-10# YOU SHOULD DO THIS LINE.
#         elif window.count(competitor_player_piece) == 4 and window.count(EMPTY) == 1:
#             score=-16# YOU SHOULD DO THIS LINE.
#         elif window.count(EMPTY)==0:
#             score=0
#         else:
#             score =window.count(competitor_player_piece)-window.count(piece)
# second
    if window.count(piece) == 5:
        score+=20
    elif window.count(piece) == 4 and window.count(EMPTY) == 1:
        score+=18
    elif window.count(piece) == 3 and window.count(EMPTY) == 2:
        score+=10
    if window.count(competitor_player_piece) == 5:
        score-=20
    elif window.count(competitor_player_piece) == 4 and window.count(EMPTY) == 1:
        score-=18
    elif window.count(competitor_player_piece) == 3 and window.count(EMPTY) == 2:
        score-=10
#     print(f"window:{window},score:{score}")
    
    return score



def score_position(board, piece):
# This function tries to return a score for the board
    score = 0
#     print(f"score position:\n{board}------------")
#     print("H-------------------")
    # Score horizontal position
    for r in range(ROW_COUNT):
        row_array = [int(i) for i in list(board[r,:])]
        for c in range(COLUMN_COUNT-4):
            window = row_array[c:c+5] # YOU SHOULD DO THIS LINE. a part of the row arrow starting from column c and ends at column c+??? 
                     # The size of this winodw should fulfil the requirement of the winning
            score += evaluate_window(list(window), piece) # YOU SHOULD DO THIS LINE. You need to add the score of this window to the score of board 
                     # that was calculated so far
#     print("V---------------------")
    # Score vertical position
    for c in range(COLUMN_COUNT):
        col_array = [int(i) for i in list(board[:,c])]
        for r in range(ROW_COUNT-4):
            window = col_array[r:r+5] # YOU SHOULD DO THIS LINE. a part of the col arrow starting from row r and ends at row r+??? 
                     # The size of this winodw should fulfil the requirement of the winning
            score += evaluate_window(list(window), piece) # YOU SHOULD DO THIS LINE. You need to add the score of this window to the score of board 
                     # that was calculated so far
#     print("D---------------------")
    # Score positively sloped diagonal position
    for r in range(ROW_COUNT-4):
        for c in range(COLUMN_COUNT-4):
#             window = np.diagonal(board[r:r+5][c:c+5]) # YOU SHOULD DO THIS LINE. a diagonal shaped part of the board starting from row r and column c and ends at row+??? and column c+??? 
                     # The size of this winodw should fulfil the requirement of the winning
            window=[board[r+i][c+i] for i in range(5)]
            score += evaluate_window(list(window), piece) # YOU SHOULD DO THIS LINE. You need to add the score of this window to the score of board 
                     # that was calculated so far
#     print("RD------------------")
    # Score negatively sloped diagonal position
#     for r in range(ROW_COUNT-4):
#         for c in range(COLUMN_COUNT-4):
#             window=np.fliplr(board)
#             window=np.diagonal(window[r:r+5][c:c+5]) # YOU SHOULD DO THIS LINE. a diagonal shaped part of the board starting from row r and column c and ends at row-??? and column c+??? 
    for c in range(COLUMN_COUNT - 4):
        for r in range(4, ROW_COUNT):               # The size of this winodw should fulfil the requirement of the winning
            window=[board[r-i][c+i] for i in range(5)]
            score += evaluate_window(list(window), piece) # YOU SHOULD DO THIS LINE. You need to add the score of this window to the score of board 
                     # that was calculated so far
#     print(f"score:{score}")
    return score


In [24]:
def maximizer(board, depth):
    
    # This part returns the column and the score (utility) if the input board is the leaf of the tree
    if is_terminal_node(board):
        if winning_move(board,PLAYER1_PIECE):# YOU SHOULD DO THIS LINE.
            return(1000)# YOU SHOULD DO THIS LINE.
        elif winning_move(board,PLAYER2_PIECE):# YOU SHOULD DO THIS LINE.
            return(-1000)# YOU SHOULD DO THIS LINE.
        else:
            return(0)
    
    # In this part the simulation ends based on the depth that was determined, but the node is not the leaf of the tree
    # so you require to use an evaluation function to estimate the score of that board
    elif depth == 0:
        return(score_position(board, PLAYER1_PIECE))# YOU SHOULD DO THIS LINE.
    
    value = -1000# YOU SHOULD DO THIS LINE.
    cols = get_valid_locations(board) # all valid columns the player can drop a piece in them
    
    for col in cols: # for each valid column the simulation must be run to find its score
            
        # for the column you need to find the row in which the piece will be dropped
        row = get_next_open_row(board, col)# YOU SHOULD DO THIS LINE. 
            
        board_copy = copy.copy(board) # not to change the board while it is simulating the following actions
        
        # YOU SHOULD DO THIS LINE. In this line you drop the piece in the column to simulate the process
        drop_piece(board_copy, row, col, PLAYER1_PIECE)
            
        score=minimizer(board_copy, depth-1)
        value = max(value,score) # the best value
    
    return value
    
    
    
def minimizer(board, depth):
    
    # This part returns the column and the score (utility) if the input board is the leaf of the tree
    if is_terminal_node(board):
        if winning_move(board,PLAYER2_PIECE):# YOU SHOULD DO THIS LINE.
            return(-1000)# YOU SHOULD DO THIS LINE.
        elif winning_move(board,PLAYER1_PIECE):# YOU SHOULD DO THIS LINE.
            return(1000)# YOU SHOULD DO THIS LINE.
        else:
            return(0)
    
    # In this part the simulation ends based on the depth that was determined, but the node is not the leaf of the tree
    # so you require to use an evaluation function to estimate the score of that board
    elif depth == 0:
        return(score_position(board, PLAYER2_PIECE))# YOU SHOULD DO THIS LINE.
    
    value = 1000# YOU SHOULD DO THIS LINE.
    cols = get_valid_locations(board) # all valid columns the player can drop a piece in them
    
    for col in cols: # for each valid column the simulation must be run to find its score
            
        # for the column you need to find the row in which the piece will be dropped
        row = get_next_open_row(board, col)# YOU SHOULD DO THIS LINE. 
            
        board_copy = copy.copy(board) # not to change the board while it is simulating the following actions
        # YOU SHOULD DO THIS LINE. In this line you drop the piece in the column to simulate the process
        drop_piece(board_copy, row, col, PLAYER2_PIECE)
        
        score=maximizer(board_copy, depth-1)
        value = min(value, score) # the best value 
    return value 
    
    
            
    
def minimax_get_action(board, depth, player):
# This function uses the minimax algorithm and return the best column that the player can drop its piece in
    
    cols = get_valid_locations(board) # all valid columns the player can drop a piece in them

    
    if player == PLAYER1: # maximizer, player1 is considered as the maximum player
        best_col = random.choice(cols) # a default value was considered as the best column, it will be changed
                                       # in the following lines
        scores = []
        for col in cols: # for each valid column the simulation must be run to find its score
            
            # for the column you need to find the row in which the piece will be dropped
            row = get_next_open_row(board, col)# YOU SHOULD DO THIS LINE.
            
            board_copy = copy.copy(board) # not to change the board while it is simulating the following actions
            
            # YOU SHOULD DO THIS LINE. In this line you drop the piece in the column to simulate the process
            drop_piece(board_copy, row, col, PLAYER1_PIECE)
            score=minimizer(board_copy, depth)
            scores.append(score) # You need to build the tree and continue the simulation
#             print(f"player:2,col:{col}, score:{score}")                                      # until the game is terminated or the depth gets to 0
                                      # until the game is terminated or the depth gets to 0
        best_col = cols[np.argmax(scores)]# ... the best column
        return best_col
    
    
    else: # minimizer, player2 is considered as the minimum player
#         value = 1000# YOU SHOULD DO THIS LINE.
        best_col = random.choice(cols) # a default value was considered as the best column, it will be changed
                                       # in the following lines
        scores = []
        for col in cols: # for each valid column the simulation must be run to find its score
            
            # for the column you need to find the row in which the piece will be dropped
            row = get_next_open_row(board, col)# YOU SHOULD DO THIS LINE. 
            
            board_copy = copy.copy(board) # not to change the board while it is simulating the following actions
            
            # YOU SHOULD DO THIS LINE. In this line you drop the piece in the column to simulate the process
            drop_piece(board_copy, row, col, PLAYER2_PIECE)
            score=maximizer(board_copy, depth)
            scores.append(score) # You need to build the tree and continue the simulation
#             print(f"player:2,col:{col}, score:{score}")                                      # until the game is terminated or the depth gets to 0
                                      # until the game is terminated or the depth gets to 0
            
        best_col = cols[np.argmin(scores)] # ... the best column
        return best_col


In [25]:
def maximizer_alpha_beta(board, depth, alpha, beta):
    
    # This part returns the column and the score (utility) if the input board is the leaf of the tree
    if is_terminal_node(board):
        if winning_move(board,PLAYER2_PIECE):
            return(-1000)
        elif winning_move(board,PLAYER1_PIECE):
            return(1000)
        else:
            return(0)
    
    # In this part the simulation ends based on the depth that was determined, but the node is not the leaf of the tree
    # so you require to use an evaluation function to estimate the score of that board
    elif depth == 0:
        return(score_position(board, PLAYER1_PIECE))
    
    value = -1000# YOU SHOULD DO THIS LINE.
    cols = get_valid_locations(board) # all valid columns the player can drop a piece in them
    
    for col in cols: # for each valid column the simulation must be run to find its score
            
        # for the column you need to find the row in which the piece will be dropped
        row = get_next_open_row(board, col)# YOU SHOULD DO THIS LINE. 
            
        board_copy = copy.copy(board) # not to change the board while it is simulating the following actions
        
        # YOU SHOULD DO THIS LINE. In this line you drop the piece in the column to simulate the process
        drop_piece(board_copy, row, col, PLAYER1_PIECE)
            
        score = minimizer_alpha_beta(board_copy, depth-1, alpha, beta)# You need to build the tree and continue the simulation until the game is terminated or the depth 
                # gets to 0
        
        value = max(value,score) # the best value
            
        if value > beta:
            return value
        alpha = max(value,alpha)# YOU SHOULD DO THIS LINE.
    return value
    
    
    
def minimizer_alpha_beta(board, depth, alpha, beta):
    
    # This part returns the column and the score (utility) if the input board is the leaf of the tree
    if is_terminal_node(board):
        if winning_move(board,PLAYER2_PIECE):
            return(-1000)
        elif winning_move(board,PLAYER1_PIECE):
            return(1000)
        else:
            return(0)
    
    # In this part the simulation ends based on the depth that was determined, but the node is not the leaf of the tree
    # so you require to use an evaluation function to estimate the score of that board
    elif depth == 0:
        return(score_position(board, PLAYER2_PIECE))
    
    value = 1000# YOU SHOULD DO THIS LINE.
    cols = get_valid_locations(board) # all valid columns the player can drop a piece in them
    
    for col in cols: # for each valid column the simulation must be run to find its score
            
        # for the column you need to find the row in which the piece will be dropped
        row = get_next_open_row(board, col)# YOU SHOULD DO THIS LINE. 
            
        board_copy = copy.copy(board) # not to change the board while it is simulating the following actions
        
        # YOU SHOULD DO THIS LINE. In this line you drop the piece in the column to simulate the process
        drop_piece(board_copy, row, col, PLAYER2_PIECE)
            
        score = maximizer_alpha_beta(board_copy, depth-1, alpha, beta)# You need to build the tree and continue the simulation until the game is terminated or the depth 
                # gets to 0
        
        value = min(value,score) # the best value 
            
        if value < alpha:
            return value
        beta = min(value,beta)# YOU SHOULD DO THIS LINE.
    return value 
    
    
            
    
def minimax_alpha_beta_get_action(board, depth, player, alpha, beta):
# This function uses the minimax algorithm and return the best column that the player can drop its piece in
            
    alpha =-1000 # YOU SHOULD DO THIS LINE.
    beta = 1000# YOU SHOULD DO THIS LINE.
    
    cols = get_valid_locations(board) # all valid columns the player can drop a piece in them

    
    if player == PLAYER1: # maximizer, player1 is considered as the maximum player
#         value = # YOU SHOULD DO THIS LINE.
        best_col = random.choice(cols) # a default value was considered as the best column, it will be changed
                                       # in the following lines
        scores = []
        for col in cols: # for each valid column the simulation must be run to find its score
            # for the column you need to find the row in which the piece will be dropped
            row = get_next_open_row(board, col)# YOU SHOULD DO THIS LINE.
            
            board_copy = copy.copy(board) # not to change the board while it is simulating the following actions
            
            # YOU SHOULD DO THIS LINE. In this line you drop the piece in the column to simulate the process
            drop_piece(board_copy, row, col, PLAYER1_PIECE)
            score=minimizer_alpha_beta(board_copy, depth, alpha, beta)
            scores.append(score) # You need to build the tree and continue the simulation
#             print(f"player:1,col:{col}, score:{score}")                                      # until the game is terminated or the depth gets to 0
            
        best_col = cols[np.argmax(scores)]# ... the best column
                
        return best_col
    
    
    else: # minimizer, player2 is considered as the minimum player
#         value = # YOU SHOULD DO THIS LINE.
        best_col = random.choice(cols) # a default value was considered as the best column, it will be changed
                                       # in the following lines
        scores = []
        for col in cols: # for each valid column the simulation must be run to find its score
            # for the column you need to find the row in which the piece will be dropped
            row = get_next_open_row(board, col)# YOU SHOULD DO THIS LINE. 
            
            board_copy = copy.copy(board) # not to change the board while it is simulating the following actions
            
            # YOU SHOULD DO THIS LINE. In this line you drop the piece in the column to simulate the process
            drop_piece(board_copy, row, col, PLAYER2_PIECE)
            score=maximizer_alpha_beta(board_copy, depth, alpha, beta)
            scores.append(score) # You need to build the tree and continue the simulation
#             print(f"player:2,col:{col}, score:{score}")                                      # until the game is terminated or the depth gets to 0
            
        best_col = cols[np.argmin(scores)]# ... the best column
        return best_col


In [26]:
# Expectimax implementation
def maximizer_expectimax(board, depth):
    
    # This part returns the column and the score (utility) if the input board is the leaf of the tree
    if is_terminal_node(board):
        if winning_move(board,PLAYER2_PIECE):
            return(-1000)
        elif winning_move(board,PLAYER1_PIECE):
            return(1000)
        else:
            return(0)
    
    # In this part the simulation ends based on the depth that was determined, but the node is not the leaf of the tree
    # so you require to use an evaluation function to estimate the score of that board
    elif depth == 0:
        return(score_position(board, PLAYER1_PIECE))
    
    value = -1000# YOU SHOULD DO THIS LINE.
    cols = get_valid_locations(board) # all valid columns the player can drop a piece in them
    
    for col in cols: # for each valid column the simulation must be run to find its score
            
        # for the column you need to find the row in which the piece will be dropped
        row = get_next_open_row(board, col)# YOU SHOULD DO THIS LINE. 
            
        board_copy = copy.copy(board) # not to change the board while it is simulating the following actions
        
        # YOU SHOULD DO THIS LINE. In this line you drop the piece in the column to simulate the process
        drop_piece(board_copy, row, col, PLAYER1_PIECE)
            
        score = chance_node_expectimax(board_copy, depth-1)# You need to build the tree and continue the simulation until the game is terminated or the depth 
                # gets to 0
        
        value = max(value,score) # the best value
    return value
    
    
    
def chance_node_expectimax(board, depth):
    
    # This part returns the column and the score (utility) if the input board is the leaf of the tree
    if is_terminal_node(board):
        if winning_move(board,PLAYER2_PIECE):
            return(-1000)
        elif winning_move(board,PLAYER1_PIECE):
            return(1000)
        else:
            return(0)
    
    # In this part the simulation ends based on the depth that was determined, but the node is not the leaf of the tree
    # so you require to use an evaluation function to estimate the score of that board
    elif depth == 0:
        return(score_position(board, PLAYER2_PIECE))
    
    cols = get_valid_locations(board) # all valid columns the player can drop a piece in them
    value=0;
    for col in cols: # for each valid column the simulation must be run to find its score
            
        # for the column you need to find the row in which the piece will be dropped
        row = get_next_open_row(board, col)# YOU SHOULD DO THIS LINE. 
            
        board_copy = copy.copy(board) # not to change the board while it is simulating the following actions
        
        # YOU SHOULD DO THIS LINE. In this line you drop the piece in the column to simulate the process
        drop_piece(board_copy, row, col, PLAYER2_PIECE)
            
        score = maximizer_expectimax(board_copy, depth-1)# You need to build the tree and continue the simulation until the game is terminated or the depth 
                # gets to 0
        value += score/len(cols)
        
    return value 
    
    
            
    
def expectimax_get_action(board, depth, player):
# This function uses the minimax algorithm and return the best column that the player can drop its piece in
    
    cols = get_valid_locations(board) # all valid columns the player can drop a piece in them

    
    if player == PLAYER1: # maximizer, player1 is considered as the maximum player
        best_col = random.choice(cols) # a default value was considered as the best column, it will be changed
                                       # in the following lines
        scores = []
        for col in cols: # for each valid column the simulation must be run to find its score
            # for the column you need to find the row in which the piece will be dropped
            row = get_next_open_row(board, col)# YOU SHOULD DO THIS LINE.
            
            board_copy = copy.copy(board) # not to change the board while it is simulating the following actions
            
            # YOU SHOULD DO THIS LINE. In this line you drop the piece in the column to simulate the process
            drop_piece(board_copy, row, col, PLAYER1_PIECE)
            score=chance_node_expectimax(board_copy, depth)
            scores.append(score) # You need to build the tree and continue the simulation
#             print(f"player:1,col:{col}, score:{score}")                                      # until the game is terminated or the depth gets to 0
            
        best_col = cols[np.argmax(scores)]# ... the best column
                
        return best_col
    
    
    else: # chance node, player2 is considered as the random player
        best_col = random.choice(cols) # a default value was considered as the best column, it will be changed
                                       # in the following lines

        return best_col

In [30]:
board = create_board()
print_board(board)
game_over = False
turn = random.randint(PLAYER1, PLAYER2)
depth=4
alpha=-1000
beta=1000
while not game_over:
    
    if len(get_valid_locations(board)) == 0:
        label = 'TIE!'
        game_over = True
    
    if turn == PLAYER1 and not game_over:
        print("player1")
#         col = random.choice(get_valid_locations(board))
#         col = minimax_get_action(board, depth, PLAYER1) # YOU SHOULD DO THIS LINE.
        col = minimax_alpha_beta_get_action(board, depth, PLAYER1, alpha, beta) # YOU SHOULD DO THIS LINE.
#         col =expectimax_get_action(board, depth, PLAYER1)
        # UNCOMMENT THIS PART AND PLAY YOURSElF
#         col = int(input('Please enter the column you want to drop in (0 to 9): '))
#         while not col in get_valid_locations(board):
#             col = int(input('Please enter the column you want to drop in (0 to 9): '))

        if is_valid_location(board, col):
            row = get_next_open_row(board, col)
            drop_piece(board, row, col, PLAYER1_PIECE)

            if winning_move(board, PLAYER1_PIECE):
                label = 'Player 1 wins!'
                game_over = True

            turn = 1 - turn
            print_board(board)
        
            
    elif turn == PLAYER2 and not game_over:
        print("player2")
#         col = random.choice(get_valid_locations(board))
#         col = minimax_get_action(board, depth, PLAYER2) # YOU SHOULD DO THIS LINE.
        col = minimax_alpha_beta_get_action(board, depth, PLAYER2, alpha, beta) # YOU SHOULD DO THIS LINE.
#         col =expectimax_get_action(board, depth, PLAYER2)
        # UNCOMMENT THIS PART AND PLAY YOURSEF
#         col = int(input('Please enter the column you want to drop in (0 to 9): '))
#         while not col in get_valid_locations(board):
#             col = int(input('Please enter the column you want to drop in (0 to 9): '))

        if is_valid_location(board, col):
            row = get_next_open_row(board, col)
            drop_piece(board, row, col, PLAYER2_PIECE)

            if winning_move(board, PLAYER2_PIECE):
                label = 'Player 2 wins!'
                game_over = True

            turn = 1 - turn
            print_board(board)
        
                   
    if game_over:
        label = label + ' Game over'
        print ('\n', colored(label, 'green'))


0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
player2

0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 
player1

0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
1 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 
player2

0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 
1 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 
player1

0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 
1 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 
1 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 
player2

0 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 
1 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 
1 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 
player1

1 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 
1 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 
1 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 
player2

1 0 0 0 0 0 0 0 0 0 
2 0 0 0 0 0 0 0 0 0 
1 0 0 0 0 0 